# Friend A — resumable dense pipeline (STEP 1c → STEP 4 → conditional STEP 5)

This is the single local entry point requested on 2026-09-10. It continues after the completed STEP 1/1b run and does
not submit anything to the Kaggle leaderboard.

`Run All` performs, in order:

1. checks `vitaliineibauer/biohub-ft-weights` for `teacher_geff/` and `train_dense_geff/`;
2. if needed, creates the exp03B + seed314159 ensemble teacher for the fixed 126 `6bba` training videos and 12 `44b6`
   diagnostic videos, builds dense GEFF labels with GT winning within 5 µm, and uploads both requested folders;
3. trains the joint detector + linker from scratch on dense `6bba`, evaluates checkpoints 5/10/15/20 on the original
   sparse-GT `44b6` holdout, uploads the official best as `friend_a_exp03_loeo_dense/`, and records STEP 4;
4. if the best score is at least 0.70, extends teacher/dense labels to all 199 videos, retrains from scratch and uploads
   `friend_a_full199_dense/`; otherwise it runs the requested 10-epoch sparse `--det-neg-weight 0.001` fallback.

The user explicitly requested 20 dense-validator epochs in this task. This overrides the 30 epochs currently written in
`TEAM.md`; the difference is recorded in `results.json` and `experiments/RESULTS.md`.

## Safe stop and resume

- Press Jupyter **Interrupt/Stop once**. The active child process is terminated.
- Teacher generation is committed locally in small chunks; only the interrupted chunk repeats.
- Dense GEFF creation is per video and validates/reuses finished outputs.
- Training saves an atomic full state after every completed epoch: model, optimizer, cosine scheduler, AMP scaler and RNG.
  The interrupted epoch repeats; finished epochs do not.
- Completed official checkpoint evaluations and completed uploads are skipped on rerun.
- Keep `outputs/local_friend_a_exp03_loeo_dense_20ep/active/` and `train_dense/` until the pipeline is complete.

Start Jupyter from `C:\Projects\CellTracker`, select the project Python 3.12 kernel, and choose **Run All**. The notebook
defaults to the full pipeline. For a dry preflight only, start Jupyter with:

```powershell
$env:BIOHUB_DENSE_RUN_PIPELINE = "0"
jupyter lab
```


In [ ]:
from __future__ import annotations

import csv
import hashlib
import importlib.util
import json
import math
import os
import platform
import re
import shutil
import subprocess
import sys
import time
import zipfile
from datetime import datetime
from pathlib import Path


EXPERIMENT = "local_friend_a_exp03_loeo_dense_20ep"
STEP4_METHOD = "friend_a_exp03_loeo_dense_20ep"
STEP5_EXPERIMENT = "local_friend_a_exp04_full199_dense_20ep"
STEP5_METHOD = "friend_a_full199_dense_20ep"
FALLBACK_EXPERIMENT = "local_friend_a_exp04_loeo_sparse_neg0001_10ep"
FALLBACK_METHOD = "friend_a_loeo_sparse_neg0001_10ep"
EXPECTED_BRANCH = "detector-baseline-v2"
WEIGHTS_DATASET = "vitaliineibauer/biohub-ft-weights"
TEACHER_DATASET_FOLDER = "teacher_geff"
DENSE_DATASET_FOLDER = "train_dense_geff"
STEP4_WEIGHTS_FOLDER = "friend_a_exp03_loeo_dense"
STEP5_WEIGHTS_FOLDER = "friend_a_full199_dense"
FALLBACK_WEIGHTS_FOLDER = "friend_a_exp04_loeo_sparse_neg0001"

STEP4_EPOCHS = 20  # explicit user override; TEAM.md currently says 30
STEP5_EPOCHS = 20
FALLBACK_EPOCHS = 10
CHECKPOINT_EVERY = 5
MILESTONE_EPOCHS = (5, 10, 15, 20)
BATCH_SIZE = 4
NUM_WORKERS = 0 if os.name == "nt" else 4
LEARNING_RATE = 1e-4
MIN_LEARNING_RATE = 1e-5
SEED = 42
WINDOW_SIZE = 2
DOWNSAMPLE = "1,4,4"
POOL_KERNEL_UM = 5.0
DET_LOSS_WEIGHT = 1.0
SPARSE_NEG_WEIGHT = 0.01
DENSE_NEG_WEIGHT = 0.01
FALLBACK_NEG_WEIGHT = 0.001
DET_THRESHOLD = 0.965
ILP_APPEARANCE = 0.0
ILP_DISAPPEARANCE = 2.0
ILP_DIVISION = 1.2
TEACHER_CHUNK_SIZE = 6
TEACHER_DEDUP_RADIUS_UM = 5.0

RUN_PIPELINE = os.environ.get("BIOHUB_DENSE_RUN_PIPELINE", "1") != "0"
AUTO_UPLOAD = os.environ.get("BIOHUB_DENSE_AUTO_UPLOAD", "1") != "0"
AUTO_GIT = os.environ.get("BIOHUB_DENSE_AUTO_GIT", "1") != "0"
CONTINUE_AFTER_STEP4 = os.environ.get("BIOHUB_DENSE_CONTINUE_AFTER_STEP4", "1") != "0"
RUN_ID = os.environ.get("BIOHUB_DENSE_RUN_ID", "active").strip() or "active"


def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        for repo_candidate in (candidate, candidate / ".github-push-worktree"):
            if (repo_candidate / "TEAM.md").exists() and (repo_candidate / "tools" / "repo" / "scripts").exists():
                return repo_candidate
    raise FileNotFoundError("Repository root not found. Start Jupyter inside C:\\Projects\\CellTracker.")


PROJECT_ROOT = find_project_root(Path.cwd())
WORKSPACE_ROOT = PROJECT_ROOT.parent if PROJECT_ROOT.name == ".github-push-worktree" else PROJECT_ROOT
GIT = ["git", "-c", f"safe.directory={PROJECT_ROOT.as_posix()}"]
DATA_DIR = Path(os.environ.get("BIOHUB_DATA_DIR", WORKSPACE_ROOT / "train")).expanduser().resolve()
DENSE_DATA_DIR = Path(os.environ.get("BIOHUB_DENSE_DATA_DIR", WORKSPACE_ROOT / "train_dense")).expanduser().resolve()
OUTPUT_ROOT = Path(os.environ.get("BIOHUB_OUTPUT_DIR", WORKSPACE_ROOT / "outputs")).expanduser().resolve()
RUN_DIR = OUTPUT_ROOT / EXPERIMENT / RUN_ID
PIPELINE_STATE_PATH = RUN_DIR / "pipeline_state.json"
REPO_DIR = RUN_DIR / "repo"
PREDICTIONS_ROOT = RUN_DIR / "predictions"
TEACHER_DIR = RUN_DIR / TEACHER_DATASET_FOLDER
EVALUATION_ROOT = RUN_DIR / "evaluation"
MANIFEST_PATH = PROJECT_ROOT / "experiments" / "_lib" / "loeo_train6bba_val44b6.json"
TRAIN_PATCH_PATH = PROJECT_ROOT / "experiments" / "_lib" / "train_patches.py"
PREDICT_PATCH_PATH = PROJECT_ROOT / "experiments" / "_lib" / "predict_patches.py"
RESUME_PATCH_PATH = PROJECT_ROOT / "experiments" / "_lib" / "full_state_resume_patch.py"
BUILDER_PATH = PROJECT_ROOT / "tools" / "build_dense_gt.py"
EXPERIMENT_DIR = PROJECT_ROOT / "experiments" / EXPERIMENT
STEP5_EXPERIMENT_DIR = PROJECT_ROOT / "experiments" / STEP5_EXPERIMENT
FALLBACK_EXPERIMENT_DIR = PROJECT_ROOT / "experiments" / FALLBACK_EXPERIMENT
REFERENCE_ROOT = OUTPUT_ROOT / "_dense_pipeline_weights"
PRIMARY_WEIGHTS_DIR = REFERENCE_ROOT / "exp03B"
SECONDARY_WEIGHTS_DIR = REFERENCE_ROOT / "seed314159"
PRIMARY_WEIGHTS = PRIMARY_WEIGHTS_DIR / "edge_predictor_best.pth"
SECONDARY_WEIGHTS = SECONDARY_WEIGHTS_DIR / "edge_predictor_best.pth"
PRIMARY_EXPECTED_SHA256 = "f3bb1932efe8fbb6893f7707eebb0355814d53f86081e0d8c48d90140e98b305"
SECONDARY_EXPECTED_SHA256 = "9bac2fa0dadc4a6fc1899e0caf187f4b553e0a7cd90ba1261a68b35ffe9e305f"
PRIMARY_DATASET_FOLDER = "exp03B"
SECONDARY_DATASET = "pilkwang/biohub-temporal-unet3d-seed314159-v1"
SECONDARY_REMOTE_FOLDER = "weights/unet_transformer/split_0"
ALL_MANIFEST_PATH = RUN_DIR / "all199.json"
TRAIN_SPLIT_PATH = RUN_DIR / "step4_train_split.json"
EVAL_SPLIT_PATH = RUN_DIR / "step4_eval_split.json"


def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(text, encoding="utf-8")
    temporary.replace(path)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def load_state() -> dict:
    if not PIPELINE_STATE_PATH.is_file():
        return {"experiment": EXPERIMENT, "run_id": RUN_ID, "status": "new"}
    return json.loads(PIPELINE_STATE_PATH.read_text(encoding="utf-8"))


def save_state(**updates) -> dict:
    state = load_state()
    state.update(updates)
    state["updated_at"] = datetime.now().astimezone().isoformat()
    atomic_write_text(PIPELINE_STATE_PATH, json.dumps(state, ensure_ascii=False, indent=2))
    return state


def terminate_process(process: subprocess.Popen) -> None:
    if process.poll() is not None:
        return
    process.terminate()
    try:
        process.wait(timeout=30)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait(timeout=30)


def stream_command(command: list[str], *, cwd: Path, env: dict, log_path: Path, label: str) -> None:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"\n[{label}] {subprocess.list2cmdline(command)}", flush=True)
    with log_path.open("a", encoding="utf-8") as log_handle:
        log_handle.write(f"\n===== {datetime.now().astimezone().isoformat()} | {label} =====\n")
        log_handle.flush()
        process = subprocess.Popen(
            command,
            cwd=cwd,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            errors="replace",
            bufsize=1,
        )
        try:
            assert process.stdout is not None
            for line in process.stdout:
                log_handle.write(line)
                log_handle.flush()
                encoding = getattr(sys.stdout, "encoding", None) or "utf-8"
                print(line.encode(encoding, errors="replace").decode(encoding, errors="replace"), end="", flush=True)
            return_code = process.wait()
        except KeyboardInterrupt:
            terminate_process(process)
            save_state(status=f"interrupted_during_{label}")
            print("\nStopped safely; Run All again to resume from the last completed unit.", flush=True)
            raise
        except BaseException:
            terminate_process(process)
            save_state(status=f"notebook_error_during_{label}")
            raise
    if return_code != 0:
        save_state(status=f"failed_during_{label}", failed_log=str(log_path), return_code=return_code)
        raise RuntimeError(f"{label} failed with exit code {return_code}; see {log_path}")


def checked(command: list[str], *, cwd: Path, env: dict | None = None) -> str:
    result = subprocess.run(
        command,
        cwd=cwd,
        env=env,
        text=True,
        encoding="utf-8",
        errors="replace",
        capture_output=True,
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed ({result.returncode}): {subprocess.list2cmdline(command)}\n"
            + (result.stdout + result.stderr)[-6000:]
        )
    return result.stdout


def import_file(module_name: str, path: Path):
    spec = importlib.util.spec_from_file_location(module_name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Cannot import {path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def common_env(data_dir: Path) -> dict[str, str]:
    env = {
        **os.environ,
        "BIOHUB_DATA_DIR": str(data_dir),
        "BIOHUB_PREDICTIONS_DIR": str(PREDICTIONS_ROOT),
        "PYTHONPATH": str(REPO_DIR / "src"),
        "PYTHONUNBUFFERED": "1",
        "PYTHONUTF8": "1",
        "PYTHONIOENCODING": "utf-8",
        "PYTORCH_ALLOC_CONF": "expandable_segments:True",
    }
    env.pop("BIOHUB_ENSEMBLE_WEIGHTS", None)
    return env


def guarded_remove_tree(path: Path, parent: Path) -> None:
    resolved = path.resolve()
    if not resolved.is_relative_to(parent.resolve()):
        raise RuntimeError(f"Refusing to remove path outside {parent}: {path}")
    if path.exists():
        shutil.rmtree(path)


def update_marked_block(path: Path, marker: str, line: str, *, heading_prefix: str | None = None) -> None:
    text_value = path.read_text(encoding="utf-8")
    block = marker + "\n" + line
    pattern = re.compile(re.escape(marker) + r"\n[^\n]*")
    if pattern.search(text_value):
        text_value = pattern.sub(block, text_value, count=1)
    elif heading_prefix:
        heading = next((item for item in text_value.splitlines() if item.startswith(heading_prefix)), None)
        if heading is None:
            raise RuntimeError(f"Heading not found in {path}: {heading_prefix}")
        text_value = text_value.replace(heading, heading + "\n" + block, 1)
    else:
        text_value = text_value.rstrip() + "\n\n" + block + "\n"
    atomic_write_text(path, text_value.rstrip() + "\n")


print(f"project:     {PROJECT_ROOT}")
print(f"sparse data: {DATA_DIR}")
print(f"dense data:  {DENSE_DATA_DIR}")
print(f"run:         {RUN_DIR}")


In [ ]:
# Validate the fixed split, prepare a patched working copy, and define reusable helpers.

required_modules = [
    "torch", "tracksdata", "geff", "polars", "rustworkx", "zarr", "tqdm",
    "scipy", "skimage", "dask", "networkx", "pandas", "pyscipopt", "ilpy", "blosc2", "kaggle",
]
missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing_modules:
    raise ImportError("Missing pipeline packages: " + ", ".join(missing_modules))

import torch

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(f"Python 3.12 is required; current interpreter: {sys.version.split()[0]}")
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Training directory is missing: {DATA_DIR}")

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
train_ids = list(manifest["train"])
holdout_ids = list(manifest["holdout"])
if len(train_ids) != 126 or len(holdout_ids) != 12:
    raise RuntimeError(f"Unexpected LOEO manifest size: {len(train_ids)} train, {len(holdout_ids)} holdout")
if set(train_ids) & set(holdout_ids):
    raise RuntimeError("Leakage: train and holdout IDs overlap")
if any(not value.startswith("6bba_") for value in train_ids):
    raise RuntimeError("Leakage: STEP 4 train side must contain only 6bba")
if any(not value.startswith("44b6_") for value in holdout_ids):
    raise RuntimeError("Leakage: STEP 4 holdout must contain only 44b6")

all_ids = sorted(path.stem for path in DATA_DIR.glob("*.geff"))
if len(all_ids) != 199 or len(set(all_ids)) != 199:
    raise RuntimeError(f"Expected 199 source GEFF datasets, found {len(all_ids)}")
missing_pairs = [
    str(DATA_DIR / f"{dataset_id}{suffix}")
    for dataset_id in all_ids
    for suffix in (".geff", ".zarr")
    if not (DATA_DIR / f"{dataset_id}{suffix}").exists()
]
if missing_pairs:
    raise FileNotFoundError("Source data are incomplete:\n" + "\n".join(missing_pairs[:20]))

RUN_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_ROOT.mkdir(parents=True, exist_ok=True)
EVALUATION_ROOT.mkdir(parents=True, exist_ok=True)
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
atomic_write_text(TRAIN_SPLIT_PATH, json.dumps([{"split": 0, "train": train_ids, "test": holdout_ids}], indent=2))
atomic_write_text(EVAL_SPLIT_PATH, json.dumps([{"split": 0, "train": [], "test": holdout_ids}], indent=2))
atomic_write_text(ALL_MANIFEST_PATH, json.dumps({"all": all_ids}, indent=2))

branch = checked([*GIT, "branch", "--show-current"], cwd=PROJECT_ROOT).strip()
if AUTO_GIT and branch != EXPECTED_BRANCH:
    raise RuntimeError(f"Auto-publish needs branch {EXPECTED_BRANCH!r}; current branch is {branch!r}")

if RUN_PIPELINE:
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is unavailable. Select the project Python 3.12 kernel and restart Jupyter.")
    cuda_probe = torch.zeros(1, device="cuda")
    del cuda_probe
    torch.cuda.empty_cache()
    cuda_device = torch.cuda.get_device_name(0)
    print(f"CUDA OK: torch={torch.__version__}, runtime={torch.version.cuda}, device={cuda_device}")
else:
    cuda_device = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "dry-run (CUDA not required)"


def prepare_repo_copy() -> None:
    marker = REPO_DIR / ".friend_a_dense_pipeline_patch_v1"
    if marker.is_file():
        return
    if REPO_DIR.exists():
        guarded_remove_tree(REPO_DIR, RUN_DIR)
    REPO_DIR.mkdir(parents=True)
    for subdir in ("scripts", "src"):
        shutil.copytree(PROJECT_ROOT / "tools" / "repo" / subdir, REPO_DIR / subdir)
    train_patch = import_file("friend_a_dense_train_patches", TRAIN_PATCH_PATH)
    predict_patch = import_file("friend_a_dense_predict_patches", PREDICT_PATCH_PATH)
    resume_patch = import_file("friend_a_dense_resume_patch", RESUME_PATCH_PATH)
    train_script = REPO_DIR / "scripts" / "train_unet_transformer.py"
    predict_script = REPO_DIR / "scripts" / "predict_unet_transformer.py"
    train_patch.apply_train_patches(train_script)
    resume_patch.apply_full_state_resume_patch(train_script)
    predict_patch.apply_predict_patches(predict_script)
    compile(train_script.read_text(encoding="utf-8"), str(train_script), "exec")
    compile(predict_script.read_text(encoding="utf-8"), str(predict_script), "exec")
    help_env = common_env(DATA_DIR)
    help_text = checked([sys.executable, str(train_script), "--help"], cwd=REPO_DIR, env=help_env)
    for flag in (
        "--resume-state", "--amp", "--no-grad-ckpt", "--seed", "--max-detections",
        "--lr-schedule", "--min-lr", "--checkpoint-every",
    ):
        if flag not in help_text:
            raise RuntimeError(f"Patched trainer is missing {flag}")
    atomic_write_text(marker, "friend_a_dense_pipeline_patch_v1\n")


def kaggle_files(dataset: str = WEIGHTS_DATASET) -> str:
    env = {**os.environ, "PYTHONUTF8": "1", "PYTHONIOENCODING": "utf-8"}
    return checked(
        [sys.executable, "-m", "kaggle", "datasets", "files", dataset, "--page-size", "1000"],
        cwd=PROJECT_ROOT,
        env=env,
    )


def copy_model_pair(source_dir: Path, target_dir: Path, expected_sha256: str) -> Path:
    source_weights = source_dir / "edge_predictor_best.pth"
    source_config = source_dir / "config.json"
    if not source_weights.is_file() or not source_config.is_file():
        raise FileNotFoundError(f"Incomplete checkpoint pair in {source_dir}")
    actual = sha256_file(source_weights).lower()
    if actual != expected_sha256.lower():
        raise RuntimeError(f"Checkpoint hash mismatch in {source_dir}: {actual}")
    target_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_weights, target_dir / source_weights.name)
    shutil.copy2(source_config, target_dir / source_config.name)
    return target_dir / source_weights.name


def download_model_file(dataset: str, remote: str, destination: Path) -> None:
    download_dir = destination.parent / "download"
    download_dir.mkdir(parents=True, exist_ok=True)
    output = checked(
        [sys.executable, "-m", "kaggle", "datasets", "download", dataset, "-f", remote,
         "-p", str(download_dir), "--unzip", "-o"],
        cwd=PROJECT_ROOT,
        env={**os.environ, "PYTHONUTF8": "1", "PYTHONIOENCODING": "utf-8"},
    )
    if output.strip():
        print(output.strip())
    matches = [path for path in download_dir.rglob(Path(remote).name) if path.is_file()]
    if not matches:
        raise FileNotFoundError(f"Kaggle download did not produce {remote}")
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(matches[0], destination)


def ensure_teacher_models() -> tuple[Path, Path]:
    if not (PRIMARY_WEIGHTS.is_file() and PRIMARY_WEIGHTS.with_name("config.json").is_file()):
        local_primary = [
            OUTPUT_ROOT / "kaggle_weights_upload_20260907_2215" / "exp03B",
            OUTPUT_ROOT / "_reference_weights" / "exp03B",
        ]
        source = next((path for path in local_primary if (path / "edge_predictor_best.pth").is_file()), None)
        if source is not None:
            copy_model_pair(source, PRIMARY_WEIGHTS_DIR, PRIMARY_EXPECTED_SHA256)
        else:
            for name in ("edge_predictor_best.pth", "config.json"):
                download_model_file(WEIGHTS_DATASET, f"{PRIMARY_DATASET_FOLDER}/{name}", PRIMARY_WEIGHTS_DIR / name)
    if not (SECONDARY_WEIGHTS.is_file() and SECONDARY_WEIGHTS.with_name("config.json").is_file()):
        local_secondary = OUTPUT_ROOT / "_reference_weights" / "reference_pilkwang_secondary_seed314159"
        if (local_secondary / "edge_predictor_best.pth").is_file():
            copy_model_pair(local_secondary, SECONDARY_WEIGHTS_DIR, SECONDARY_EXPECTED_SHA256)
        else:
            for name in ("edge_predictor_best.pth", "config.json"):
                download_model_file(SECONDARY_DATASET, f"{SECONDARY_REMOTE_FOLDER}/{name}", SECONDARY_WEIGHTS_DIR / name)
    for path, expected in ((PRIMARY_WEIGHTS, PRIMARY_EXPECTED_SHA256), (SECONDARY_WEIGHTS, SECONDARY_EXPECTED_SHA256)):
        if sha256_file(path).lower() != expected.lower():
            raise RuntimeError(f"Teacher model hash mismatch: {path}")
    return PRIMARY_WEIGHTS, SECONDARY_WEIGHTS


prepare_repo_copy()
run_config = {
    "experiment": EXPERIMENT,
    "branch": branch,
    "project_root": str(PROJECT_ROOT),
    "data_dir": str(DATA_DIR),
    "dense_data_dir": str(DENSE_DATA_DIR),
    "python": sys.version,
    "platform": platform.platform(),
    "cuda_device": cuda_device,
    "manifest_sha256": sha256_file(MANIFEST_PATH),
    "train_videos": train_ids,
    "holdout_videos": holdout_ids,
    "all_videos": all_ids,
    "step4_epochs": STEP4_EPOCHS,
    "team_step4_epochs": 30,
    "epoch_override": "explicit user request on 2026-09-10",
    "batch_size": BATCH_SIZE,
    "lr": LEARNING_RATE,
    "lr_schedule": "cosine",
    "min_lr": MIN_LEARNING_RATE,
    "seed": SEED,
    "window_size": WINDOW_SIZE,
    "downsample": [1, 4, 4],
    "step4_max_detections": 0,
    "checkpoint_every": CHECKPOINT_EVERY,
    "leaderboard_submission": False,
}
fingerprint = hashlib.sha256(json.dumps(run_config, sort_keys=True).encode()).hexdigest()
run_config["fingerprint"] = fingerprint
run_config_path = RUN_DIR / "run_config.json"
if run_config_path.is_file():
    previous = json.loads(run_config_path.read_text(encoding="utf-8"))
    if previous.get("fingerprint") != fingerprint:
        raise RuntimeError("The active run has a different recipe. Set BIOHUB_DENSE_RUN_ID to a new value.")
else:
    atomic_write_text(run_config_path, json.dumps(run_config, ensure_ascii=False, indent=2))
save_state(status="prepared", config_fingerprint=fingerprint)
print(f"Preflight OK: 126 train 6bba | 12 holdout 44b6 | {len(all_ids)} total")
if not RUN_PIPELINE:
    print("Dry preflight only: BIOHUB_DENSE_RUN_PIPELINE=0, no teacher/training/upload was started.")


In [ ]:
# Resumable teacher, training, evaluation, reporting, and Kaggle-dataset helpers.

from geff import GeffMetadata

TRAIN_SCRIPT = REPO_DIR / "scripts" / "train_unet_transformer.py"
PREDICT_SCRIPT = REPO_DIR / "scripts" / "predict_unet_transformer.py"


def upsert_results_row(key: str, line: str) -> None:
    path = PROJECT_ROOT / "experiments" / "RESULTS.md"
    lines = [value for value in path.read_text(encoding="utf-8").splitlines() if not value.startswith(key)]
    insert_at = next((index for index, value in enumerate(lines) if value.startswith("## Per-video")), len(lines))
    lines.insert(insert_at, line)
    atomic_write_text(path, "\n".join(lines).rstrip() + "\n")


def commit_and_push(paths: list[Path], message: str) -> None:
    relative = [str(path.resolve().relative_to(PROJECT_ROOT.resolve())).replace("\\", "/") for path in paths]
    already_staged = checked([*GIT, "diff", "--cached", "--name-only"], cwd=PROJECT_ROOT).splitlines()
    unexpected_staged = set(already_staged) - set(relative)
    if unexpected_staged:
        raise RuntimeError("Refusing to include already-staged unrelated files: " + ", ".join(sorted(unexpected_staged)))
    checked([*GIT, "add", "--", *relative], cwd=PROJECT_ROOT)
    staged = subprocess.run([*GIT, "diff", "--cached", "--quiet"], cwd=PROJECT_ROOT)
    if staged.returncode == 1:
        print(checked([*GIT, "commit", "-m", message], cwd=PROJECT_ROOT).strip())
    elif staged.returncode != 0:
        raise RuntimeError("git diff --cached failed")
    print(checked([*GIT, "push", "origin", EXPECTED_BRANCH], cwd=PROJECT_ROOT).strip())


def restore_geff_folder(snapshot: Path, folder: str, destination: Path, expected_ids: list[str]) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    for index, dataset_id in enumerate(expected_ids, start=1):
        target = destination / f"{dataset_id}.geff"
        if target.exists():
            continue
        candidates = [path for path in snapshot.rglob(f"{dataset_id}.geff") if path.is_dir() and folder in path.parts]
        if candidates:
            shutil.copytree(candidates[0], target)
        else:
            archives = [path for path in snapshot.rglob(f"{dataset_id}.geff.zip") if folder in path.parts]
            if not archives:
                raise FileNotFoundError(f"Remote folder {folder} has no {dataset_id}.geff(.zip)")
            unpack = RUN_DIR / "restore_tmp" / folder / dataset_id
            if unpack.exists():
                guarded_remove_tree(unpack, RUN_DIR)
            unpack.mkdir(parents=True)
            shutil.unpack_archive(str(archives[0]), str(unpack))
            nested = [path for path in unpack.rglob(f"{dataset_id}.geff") if path.is_dir()]
            source = nested[0] if nested else unpack
            shutil.copytree(source, target)
        if index % 20 == 0 or index == len(expected_ids):
            print(f"restored {folder}: {index}/{len(expected_ids)}", flush=True)


def download_dataset_snapshot() -> Path:
    snapshot = RUN_DIR / "weights_dataset_snapshot"
    marker = snapshot / ".download_complete"
    if marker.is_file():
        return snapshot
    if snapshot.exists():
        guarded_remove_tree(snapshot, RUN_DIR)
    snapshot.mkdir(parents=True)
    stream_command(
        [sys.executable, "-m", "kaggle", "datasets", "download", WEIGHTS_DATASET,
         "-p", str(snapshot), "--unzip", "-o"],
        cwd=PROJECT_ROOT,
        env={**os.environ, "PYTHONUTF8": "1", "PYTHONIOENCODING": "utf-8"},
        log_path=RUN_DIR / "dataset_download.log",
        label="download_weights_dataset",
    )
    atomic_write_text(marker, "complete\n")
    return snapshot


def stage_current_weights_dataset(stage_name: str) -> Path:
    stage = RUN_DIR / stage_name
    if stage.exists():
        guarded_remove_tree(stage, RUN_DIR)
    stage.mkdir(parents=True)
    stream_command(
        [sys.executable, "-m", "kaggle", "datasets", "download", WEIGHTS_DATASET,
         "-p", str(stage), "--unzip", "-o"],
        cwd=PROJECT_ROOT,
        env={**os.environ, "PYTHONUTF8": "1", "PYTHONIOENCODING": "utf-8"},
        log_path=RUN_DIR / f"{stage_name}.log",
        label=f"stage_{stage_name}",
    )
    atomic_write_text(
        stage / "dataset-metadata.json",
        json.dumps({"id": WEIGHTS_DATASET, "title": "biohub-ft-weights"}, indent=2) + "\n",
    )
    return stage


def publish_dataset_stage(stage: Path, message: str, expected_prefixes: list[str]) -> None:
    stream_command(
        [sys.executable, "-m", "kaggle", "datasets", "version", "-p", str(stage),
         "--dir-mode", "zip", "-m", message],
        cwd=PROJECT_ROOT,
        env={**os.environ, "PYTHONUTF8": "1", "PYTHONIOENCODING": "utf-8"},
        log_path=RUN_DIR / "dataset_upload.log",
        label="upload_weights_dataset_version",
    )
    for attempt in range(12):
        listing = kaggle_files()
        if all(prefix in listing for prefix in expected_prefixes):
            print("Kaggle dataset version is visible and contains all requested folders.")
            return
        if attempt < 11:
            print("Kaggle is still processing the dataset version; retrying in 10 seconds...")
            time.sleep(10)
    raise RuntimeError("Kaggle accepted the version but expected folders are not visible")


def upload_geff_folders(folder_sources: dict[str, Path], expected_ids: list[str], message: str) -> None:
    stage = stage_current_weights_dataset("geff_upload_stage")
    for folder, source in folder_sources.items():
        target = stage / folder
        if target.exists():
            guarded_remove_tree(target, stage)
        target.mkdir(parents=True)
        for dataset_id in expected_ids:
            source_geff = source / f"{dataset_id}.geff"
            if not source_geff.is_dir():
                raise FileNotFoundError(source_geff)
            shutil.copytree(source_geff, target / source_geff.name)
    publish_dataset_stage(stage, message, [f"{folder}/" for folder in folder_sources])


def upload_weight_pair(weights: Path, folder: str, message: str) -> None:
    state = load_state()
    state_key = f"uploaded_{folder}"
    if state.get(state_key) == sha256_file(weights):
        print(f"Already uploaded and recorded: {folder}/")
        return
    config_path = weights.with_name("config.json")
    if not weights.is_file() or not config_path.is_file():
        raise FileNotFoundError(f"Incomplete weight/config pair: {weights}")
    stage = stage_current_weights_dataset("weights_upload_stage")
    target = stage / folder
    if target.exists():
        guarded_remove_tree(target, stage)
    target.mkdir(parents=True)
    shutil.copy2(weights, target / "edge_predictor_best.pth")
    shutil.copy2(config_path, target / "config.json")
    publish_dataset_stage(stage, message, [f"{folder}/edge_predictor_best.pth", f"{folder}/config.json"])
    save_state(**{state_key: sha256_file(weights)})


def prediction_output(user: str, method: str) -> Path:
    return PREDICTIONS_ROOT / user / method / "split_0"


def predict_teacher_chunk(dataset_ids: list[str], label: str) -> list[Path]:
    primary, secondary = ensure_teacher_models()
    split_path = RUN_DIR / "teacher_splits" / f"{label}.json"
    atomic_write_text(split_path, json.dumps([{"split": 0, "train": [], "test": dataset_ids}], indent=2))
    user = "friend_a_dense_teacher"
    method = label
    env = common_env(DATA_DIR)
    env["USER"] = user
    env["BIOHUB_ENSEMBLE_WEIGHTS"] = str(secondary)
    stream_command(
        [
            sys.executable, str(PREDICT_SCRIPT), "--method", method, "--data-dir", str(DATA_DIR),
            "--splits", str(split_path), "--split", "0", "--weights", str(primary),
            "--unet-batch-size", "4", "--det-threshold", str(DET_THRESHOLD),
        ],
        cwd=REPO_DIR,
        env=env,
        log_path=RUN_DIR / "teacher_logs" / f"{label}.log",
        label=label,
    )
    output = prediction_output(user, method)
    geffs = sorted(output.glob("*.geff"))
    if {path.stem for path in geffs} != set(dataset_ids):
        raise RuntimeError(f"Teacher chunk {label} did not produce exact requested coverage")
    return geffs


def ensure_teacher(ids: list[str], *, smoke_first: bool) -> None:
    TEACHER_DIR.mkdir(parents=True, exist_ok=True)
    if smoke_first and not load_state().get("teacher_smoke_complete"):
        smoke_geffs = predict_teacher_chunk(ids[:2], "teacher_smoke2")
        for path in smoke_geffs:
            loaded = __import__("tracksdata").graph.IndexedRXGraph.from_geff(path)
            graph = loaded[0] if isinstance(loaded, tuple) else loaded
            if graph.num_nodes() <= 0:
                raise RuntimeError(f"Teacher smoke output is empty: {path}")
        guarded_remove_tree(prediction_output("friend_a_dense_teacher", "teacher_smoke2"), PREDICTIONS_ROOT)
        save_state(teacher_smoke_complete=True)
        print("Teacher smoke passed on 2 videos.")
    missing = [dataset_id for dataset_id in ids if not (TEACHER_DIR / f"{dataset_id}.geff").is_dir()]
    for offset in range(0, len(missing), TEACHER_CHUNK_SIZE):
        chunk = missing[offset: offset + TEACHER_CHUNK_SIZE]
        label = f"teacher_chunk_{offset // TEACHER_CHUNK_SIZE:03d}"
        geffs = predict_teacher_chunk(chunk, label)
        for source in geffs:
            target = TEACHER_DIR / source.name
            if target.exists():
                guarded_remove_tree(target, TEACHER_DIR)
            shutil.move(str(source), str(target))
        output_parent = prediction_output("friend_a_dense_teacher", label)
        if output_parent.exists():
            guarded_remove_tree(output_parent, PREDICTIONS_ROOT)
        save_state(status="teacher_in_progress", teacher_complete_count=sum((TEACHER_DIR / f"{x}.geff").is_dir() for x in ids))
    if any(not (TEACHER_DIR / f"{dataset_id}.geff").is_dir() for dataset_id in ids):
        raise RuntimeError("Teacher generation ended with missing datasets")


def training_output(method: str) -> Path:
    return REPO_DIR / "weights" / method / "split_0"


def fixed_train_command(
    *, method: str, data_dir: Path, split_path: Path, epochs: int, neg_weight: float,
    max_detections: int, max_iters: int | None = None,
) -> list[str]:
    command = [
        sys.executable, str(TRAIN_SCRIPT), "--method", method, "--data-dir", str(data_dir),
        "--splits", str(split_path), "--split", "0", "--epochs", str(epochs),
        "--batch-size", str(BATCH_SIZE), "--num-workers", str(NUM_WORKERS),
        "--lr", str(LEARNING_RATE), "--lr-schedule", "cosine", "--min-lr", str(MIN_LEARNING_RATE),
        "--det-loss", "bce", "--det-loss-weight", str(DET_LOSS_WEIGHT),
        "--det-neg-weight", str(neg_weight), "--window-size", str(WINDOW_SIZE),
        "--downsample", DOWNSAMPLE, "--pool-kernel-um", str(POOL_KERNEL_UM),
        "--max-detections", str(max_detections), "--checkpoint-every", str(CHECKPOINT_EVERY),
        "--seed", str(SEED), "--amp", "--no-grad-ckpt", "--single-gpu",
    ]
    if max_iters is not None:
        command.extend(["--max-iters", str(max_iters)])
    if "--init-weights" in command or "--unet-weights" in command:
        raise RuntimeError("Scratch-training leakage guard failed")
    return command


def smoke_training(label: str, data_dir: Path, ids: list[str], neg_weight: float, max_detections: int) -> None:
    key = f"{label}_smoke_complete"
    if load_state().get(key):
        print(f"Skipping completed smoke: {label}")
        return
    split = RUN_DIR / "smoke_splits" / f"{label}.json"
    atomic_write_text(split, json.dumps([{"split": 0, "train": ids[:2], "test": ids[:2]}], indent=2))
    command = fixed_train_command(
        method=f"{label}_smoke", data_dir=data_dir, split_path=split, epochs=1,
        neg_weight=neg_weight, max_detections=max_detections, max_iters=2,
    )
    stream_command(command, cwd=REPO_DIR, env=common_env(data_dir), log_path=RUN_DIR / f"{label}_smoke.log", label=f"{label}_smoke")
    output = training_output(f"{label}_smoke")
    if not (output / "edge_predictor_best.pth").is_file():
        raise RuntimeError(f"Smoke test produced no checkpoint: {label}")
    save_state(**{key: True})
    print(f"Smoke passed: {label}")


def run_training(
    label: str, method: str, data_dir: Path, train_set: list[str], test_set: list[str],
    epochs: int, neg_weight: float, max_detections: int,
) -> Path:
    split = RUN_DIR / "training_splits" / f"{label}.json"
    atomic_write_text(split, json.dumps([{"split": 0, "train": train_set, "test": test_set}], indent=2))
    output = training_output(method)
    state_path = output / "training_state_latest.pt"
    completed = 0
    if state_path.is_file():
        payload = torch.load(state_path, map_location="cpu", weights_only=False)
        if payload.get("format") != "biohub_full_training_state_v1":
            raise RuntimeError(f"Unknown resume state format: {state_path}")
        if int(payload.get("target_epochs", -1)) != epochs:
            raise RuntimeError(f"Resume state target differs from requested {epochs} epochs")
        completed = int(payload["completed_epochs"])
        del payload
    if completed < epochs:
        command = fixed_train_command(
            method=method, data_dir=data_dir, split_path=split, epochs=epochs,
            neg_weight=neg_weight, max_detections=max_detections,
        )
        if completed:
            command.extend(["--resume-state", str(state_path)])
            print(f"Resuming {label} after epoch {completed}/{epochs}")
        else:
            print(f"Starting {label} from scratch")
        atomic_write_text(RUN_DIR / f"{label}_command.txt", subprocess.list2cmdline(command) + "\n")
        save_state(status=f"training_{label}", active_training=label, completed_epochs_before_launch=completed)
        stream_command(command, cwd=REPO_DIR, env=common_env(data_dir), log_path=RUN_DIR / f"{label}_train.log", label=f"train_{label}")
    payload = torch.load(state_path, map_location="cpu", weights_only=False)
    if int(payload.get("completed_epochs", -1)) != epochs:
        raise RuntimeError(f"{label} returned before completing {epochs} epochs")
    del payload
    save_state(status=f"training_{label}_complete", active_training=None)
    return output


def metric_node_counts(rows: list[dict]) -> tuple[int, float, float]:
    predicted = 0
    estimated = 0.0
    for row in rows:
        predicted += int(row["num_pred_nodes"])
        metadata = GeffMetadata.read(DATA_DIR / f"{row['dataset']}.geff")
        value = float((metadata.extra or {}).get("estimated_number_of_nodes", float("nan")))
        if not math.isfinite(value) or value <= 0:
            raise RuntimeError(f"No estimated_number_of_nodes for {row['dataset']}")
        estimated += value
    return predicted, estimated, predicted / estimated


def evaluate_checkpoint(label: str, weights: Path, expected_ids: list[str]) -> dict:
    evaluation = EVALUATION_ROOT / label
    metrics_path = evaluation / "official_metrics.json"
    if metrics_path.is_file():
        existing = json.loads(metrics_path.read_text(encoding="utf-8"))
        if int(existing.get("summary", {}).get("n", -1)) == len(expected_ids) and existing.get("prediction_zip"):
            if Path(existing["prediction_zip"]).is_file():
                print(f"Skipping completed official evaluation: {label}")
                return existing
    evaluation.mkdir(parents=True, exist_ok=True)
    if not weights.is_file() or not weights.with_name("config.json").is_file():
        raise FileNotFoundError(f"Checkpoint/config missing: {weights}")
    user = "friend_a_dense_eval"
    method = label
    env = common_env(DATA_DIR)
    env["USER"] = user
    stream_command(
        [
            sys.executable, str(PREDICT_SCRIPT), "--method", method, "--data-dir", str(DATA_DIR),
            "--splits", str(EVAL_SPLIT_PATH), "--split", "0", "--weights", str(weights),
            "--unet-batch-size", "4", "--det-threshold", str(DET_THRESHOLD), "--use-ilp",
            "--ilp-appearance-weight", str(ILP_APPEARANCE),
            "--ilp-disappearance-weight", str(ILP_DISAPPEARANCE),
            "--ilp-division-weight", str(ILP_DIVISION),
        ],
        cwd=REPO_DIR,
        env=env,
        log_path=evaluation / "predict.log",
        label=f"predict_{label}",
    )
    prediction_dir = prediction_output(user, method)
    geffs = sorted(prediction_dir.glob("*.geff"))
    if {path.stem for path in geffs} != set(expected_ids):
        raise RuntimeError(f"Official evaluation coverage mismatch for {label}")
    stream_command(
        [sys.executable, str(PROJECT_ROOT / "tools" / "score_geffs.py"), str(prediction_dir), "--out", str(metrics_path)],
        cwd=PROJECT_ROOT,
        env={**os.environ, "BIOHUB_DATA_DIR": str(DATA_DIR), "PYTHONUTF8": "1", "PYTHONIOENCODING": "utf-8"},
        log_path=evaluation / "score.log",
        label=f"score_{label}",
    )
    payload = json.loads(metrics_path.read_text(encoding="utf-8"))
    if int(payload.get("summary", {}).get("n", -1)) != len(expected_ids):
        raise RuntimeError(f"Official scorer skipped one or more videos for {label}")
    n_pred, n_est, ratio = metric_node_counts(payload["rows"])
    zip_dir = RUN_DIR / "prediction_zips"
    zip_dir.mkdir(parents=True, exist_ok=True)
    temp_base = zip_dir / f".{label}.building"
    archive = Path(shutil.make_archive(str(temp_base), "zip", root_dir=prediction_dir))
    zip_path = zip_dir / f"{label}.zip"
    archive.replace(zip_path)
    guarded_remove_tree(prediction_dir, PREDICTIONS_ROOT)
    payload["checkpoint"] = {"label": label, "weights": str(weights), "weights_sha256": sha256_file(weights)}
    payload["aggregate_node_counts"] = {"n_pred": n_pred, "n_est": n_est, "n_pred_over_n_est": ratio}
    payload["prediction_zip"] = str(zip_path)
    atomic_write_text(metrics_path, json.dumps(payload, ensure_ascii=False, indent=2))
    print(
        f"{label}: score={payload['summary']['score']:.4f}, edge={payload['summary']['edge_jaccard']:.4f}, "
        f"recall={payload['summary']['node_recall']:.4f}, Npred/Nest={ratio:.3f}"
    )
    return payload


def evaluate_milestones(label_prefix: str, output: Path, epochs: tuple[int, ...]) -> dict[str, dict]:
    specs = [(f"epoch_{epoch:03d}", output / f"edge_predictor_epoch_{epoch:03d}.pth") for epoch in epochs]
    specs.append(("training_best", output / "edge_predictor_best.pth"))
    results = {}
    for checkpoint_label, weights in specs:
        results[checkpoint_label] = evaluate_checkpoint(f"{label_prefix}_{checkpoint_label}", weights, holdout_ids)
    return results


def compact_metric(checkpoint_label: str, payload: dict) -> dict:
    summary = payload["summary"]
    counts = payload["aggregate_node_counts"]
    epoch_match = re.search(r"epoch_(\d+)$", checkpoint_label)
    return {
        "label": checkpoint_label,
        "epoch": int(epoch_match.group(1)) if epoch_match else None,
        "score": float(summary["score"]),
        "adj_edge_jaccard": float(summary["adj_edge_jaccard"]),
        "edge_jaccard": float(summary["edge_jaccard"]),
        "division_jaccard": float(summary["division_jaccard"]),
        "division_tp": int(summary["division_tp"]),
        "division_fp": int(summary["division_fp"]),
        "division_fn": int(summary["division_fn"]),
        "node_recall": float(summary["node_recall"]),
        "n_pred": int(counts["n_pred"]),
        "n_est": float(counts["n_est"]),
        "n_pred_over_n_est": float(counts["n_pred_over_n_est"]),
        "weights": payload["checkpoint"]["weights"],
        "weights_sha256": payload["checkpoint"]["weights_sha256"],
    }


In [ ]:
# STEP 1c: obtain/create the 138-video teacher, build dense labels, publish and record the step.

step1c_ids = train_ids + holdout_ids

if RUN_PIPELINE and not load_state().get("step1c_complete"):
    listing = kaggle_files()
    remote_teacher = f"{TEACHER_DATASET_FOLDER}/" in listing
    remote_dense = f"{DENSE_DATASET_FOLDER}/" in listing
    print(f"Kaggle dataset: teacher={remote_teacher}, dense={remote_dense}")

    if remote_teacher or remote_dense:
        snapshot = download_dataset_snapshot()
        if remote_teacher:
            restore_geff_folder(snapshot, TEACHER_DATASET_FOLDER, TEACHER_DIR, step1c_ids)
        if remote_dense:
            restore_geff_folder(snapshot, DENSE_DATASET_FOLDER, DENSE_DATA_DIR, step1c_ids)

    if not remote_teacher:
        ensure_teacher(step1c_ids, smoke_first=True)

    # A two-video builder smoke precedes the full CPU build. Finished dense GEFFs are validated and reused.
    smoke_output = RUN_DIR / "dense_builder_smoke"
    smoke_stats = smoke_output / "dense_stats.csv"
    if not load_state().get("dense_builder_smoke_complete"):
        stream_command(
            [
                sys.executable, str(BUILDER_PATH), "--data-dir", str(DATA_DIR),
                "--teacher-dir", str(TEACHER_DIR), "--output-dir", str(smoke_output),
                "--manifest", str(MANIFEST_PATH), "--manifest-keys", "train,holdout",
                "--dedup-radius-um", str(TEACHER_DEDUP_RADIUS_UM),
                "--stats-out", str(smoke_stats), "--limit", "2",
            ],
            cwd=PROJECT_ROOT,
            env=common_env(DATA_DIR),
            log_path=RUN_DIR / "dense_builder_smoke.log",
            label="dense_builder_smoke2",
        )
        save_state(dense_builder_smoke_complete=True)

    tracked_stats = EXPERIMENT_DIR / "dense_stats.csv"
    stream_command(
        [
            sys.executable, str(BUILDER_PATH), "--data-dir", str(DATA_DIR),
            "--teacher-dir", str(TEACHER_DIR), "--output-dir", str(DENSE_DATA_DIR),
            "--manifest", str(MANIFEST_PATH), "--manifest-keys", "train,holdout",
            "--dedup-radius-um", str(TEACHER_DEDUP_RADIUS_UM), "--stats-out", str(tracked_stats),
        ],
        cwd=PROJECT_ROOT,
        env=common_env(DATA_DIR),
        log_path=RUN_DIR / "dense_builder.log",
        label="build_dense_138",
    )
    with tracked_stats.open(newline="", encoding="utf-8") as handle:
        dense_rows = list(csv.DictReader(handle))
    if len(dense_rows) != len(step1c_ids):
        raise RuntimeError(f"Dense stats coverage is {len(dense_rows)}/{len(step1c_ids)}")
    if any(float(row["gt_node_recall_in_dense"]) != 1.0 for row in dense_rows):
        raise RuntimeError("Dense GT recall is not exactly 1.0")

    if AUTO_UPLOAD and not (remote_teacher and remote_dense):
        upload_geff_folders(
            {TEACHER_DATASET_FOLDER: TEACHER_DIR, DENSE_DATASET_FOLDER: DENSE_DATA_DIR},
            step1c_ids,
            f"Friend A STEP 1c: teacher and dense GEFF labels for {len(step1c_ids)} LOEO videos",
        )
    elif not AUTO_UPLOAD:
        print("BIOHUB_DENSE_AUTO_UPLOAD=0: teacher/dense upload skipped by request.")

    ratios = [float(row["n_dense_over_estimated"]) for row in dense_rows]
    recalls = [float(row["teacher_recall_of_gt_5um"]) for row in dense_rows]
    result = {
        "experiment": EXPERIMENT,
        "step": "1c",
        "status": "complete",
        "completed_at": datetime.now().astimezone().isoformat(),
        "videos": len(step1c_ids),
        "train_6bba": len(train_ids),
        "diagnostic_44b6": len(holdout_ids),
        "teacher": "exp03B + pilkwang seed314159 logits ensemble",
        "teacher_det_threshold": DET_THRESHOLD,
        "teacher_use_ilp": False,
        "dense_rule": "teacher union GT; GT wins within 5 um; GT edges only",
        "teacher_recall_of_gt_5um_mean": sum(recalls) / len(recalls),
        "n_dense_over_estimated_mean": sum(ratios) / len(ratios),
        "n_dense_over_estimated_min": min(ratios),
        "n_dense_over_estimated_max": max(ratios),
        "ratio_outliers": sum(str(row["ratio_outside_0.9_1.1"]).lower() == "true" for row in dense_rows),
        "stats_csv": str(tracked_stats.relative_to(PROJECT_ROOT)).replace("\\", "/"),
        "dataset": WEIGHTS_DATASET,
        "dataset_folders": [TEACHER_DATASET_FOLDER, DENSE_DATASET_FOLDER],
        "uploaded": bool(AUTO_UPLOAD or (remote_teacher and remote_dense)),
        "leaderboard_submission": False,
    }
    step1c_result_path = EXPERIMENT_DIR / "step1c_results.json"
    atomic_write_text(step1c_result_path, json.dumps(result, ensure_ascii=False, indent=2))
    upsert_results_row(
        "| local Friend A exp03 STEP 1c |",
        f"| local Friend A exp03 STEP 1c | exp03B+seed314159 teacher (det {DET_THRESHOLD}, no ILP) -> dense GT union | - | "
        f"138 videos; teacher recall@5um mean {result['teacher_recall_of_gt_5um_mean']:.4f}; "
        f"dense/Nest mean {result['n_dense_over_estimated_mean']:.3f} | teacher GPU + CPU build | "
        f"GT recall 1.0; folders `{TEACHER_DATASET_FOLDER}/`, `{DENSE_DATASET_FOLDER}/`; no LB |",
    )
    results_md = PROJECT_ROOT / "experiments" / "RESULTS.md"
    context_path = PROJECT_ROOT / "CONTEXT.md"
    update_marked_block(
        context_path,
        "<!-- friend-a-step1c-dense-result -->",
        f"- {datetime.now().date().isoformat()} Friend A STEP 1c complete: exp03B+seed314159 teacher and dense labels "
        f"for 126 train 6bba + 12 diagnostic 44b6; teacher recall@5um mean "
        f"{result['teacher_recall_of_gt_5um_mean']:.4f}, dense/N_est mean {result['n_dense_over_estimated_mean']:.3f}; "
        f"dataset folders `{TEACHER_DATASET_FOLDER}/` and `{DENSE_DATASET_FOLDER}/`; 44b6 pseudo-points are not used for training.",
        heading_prefix="## 14. Operational state",
    )
    save_state(status="step1c_reports_written", step1c_complete=True)
    if AUTO_GIT:
        commit_and_push(
            [BUILDER_PATH, tracked_stats, step1c_result_path, results_md, context_path],
            "Complete Friend A STEP 1c dense labels",
        )
    print("STEP 1c complete. No leaderboard submission was made.")
elif RUN_PIPELINE:
    print("Skipping completed STEP 1c.")
else:
    print("Dry preflight: STEP 1c was not started.")


In [ ]:
# STEP 4: scratch dense-label LOEO training, official milestone evaluation, upload, and report.


def save_training_curves(log_path: Path, experiment_dir: Path, epochs: int) -> tuple[Path, Path | None]:
    pattern = re.compile(
        r"Epoch\s+(?P<epoch>\d+)/(?P<epochs>\d+).*?lr=(?P<lr>[0-9.eE+-]+).*?"
        r"edge=(?P<edge>[0-9.eE+-]+).*?det=(?P<det>[0-9.eE+-]+).*?"
        r"test_loss=(?P<test_loss>[0-9.eE+-]+).*?acc=(?P<acc>[0-9.eE+-]+).*?"
        r"recall=(?P<recall>[0-9.eE+-]+).*?best=(?P<best>[0-9.eE+-]+)"
    )
    by_epoch = {}
    for match in pattern.finditer(log_path.read_text(encoding="utf-8", errors="replace")):
        number = int(match.group("epoch"))
        by_epoch[number] = {
            "epoch": number,
            **{name: float(match.group(name)) for name in ("lr", "edge", "det", "test_loss", "acc", "recall", "best")},
        }
    rows = [by_epoch[number] for number in sorted(by_epoch)]
    if [row["epoch"] for row in rows] != list(range(1, epochs + 1)):
        raise RuntimeError(f"Could not reconstruct complete 1..{epochs} history from {log_path}")
    csv_path = experiment_dir / "training_history.csv"
    with csv_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    plot_path = experiment_dir / "learning_curves.png"
    try:
        import matplotlib.pyplot as plt

        figure, axes = plt.subplots(1, 2, figsize=(11, 4))
        axes[0].plot([row["epoch"] for row in rows], [row["edge"] for row in rows], label="edge loss")
        axes[0].plot([row["epoch"] for row in rows], [row["det"] for row in rows], label="detection loss")
        axes[0].plot([row["epoch"] for row in rows], [row["test_loss"] for row in rows], label="internal val loss")
        axes[0].set(xlabel="epoch", title="Losses")
        axes[0].legend()
        axes[1].plot([row["epoch"] for row in rows], [row["acc"] for row in rows], label="internal edge accuracy")
        axes[1].plot([row["epoch"] for row in rows], [row["recall"] for row in rows], label="internal node recall")
        axes[1].set(xlabel="epoch", ylim=(0, 1.02), title="Internal diagnostics")
        axes[1].legend()
        figure.tight_layout()
        figure.savefig(plot_path, dpi=140)
        plt.close(figure)
    except ImportError:
        plot_path = None
    return csv_path, plot_path


if RUN_PIPELINE and not load_state().get("step4_complete"):
    if not load_state().get("step1c_complete"):
        raise RuntimeError("STEP 1c is not complete; STEP 4 cannot start")
    missing_dense = [dataset_id for dataset_id in train_ids + holdout_ids if not (DENSE_DATA_DIR / f"{dataset_id}.geff").is_dir()]
    if missing_dense:
        raise FileNotFoundError(f"Dense dataset is missing {len(missing_dense)} GEFFs")

    smoke_training("step4_dense", DENSE_DATA_DIR, train_ids, DENSE_NEG_WEIGHT, 0)
    step4_output = run_training(
        "step4_dense", STEP4_METHOD, DENSE_DATA_DIR, train_ids, holdout_ids,
        STEP4_EPOCHS, DENSE_NEG_WEIGHT, 0,
    )
    checkpoint_paths = [step4_output / f"edge_predictor_epoch_{epoch:03d}.pth" for epoch in MILESTONE_EPOCHS]
    missing_checkpoints = [str(path) for path in checkpoint_paths if not path.is_file()]
    if missing_checkpoints:
        raise FileNotFoundError("STEP 4 checkpoints are missing:\n" + "\n".join(missing_checkpoints))
    raw_metrics = evaluate_milestones("step4_dense", step4_output, MILESTONE_EPOCHS)
    metrics = {label: compact_metric(label, payload) for label, payload in raw_metrics.items()}
    best_label = max(metrics, key=lambda label: (metrics[label]["score"], metrics[label]["epoch"] or -1))
    best = metrics[best_label]
    best_weights = Path(best["weights"])

    if AUTO_UPLOAD:
        upload_weight_pair(
            best_weights,
            STEP4_WEIGHTS_FOLDER,
            f"Friend A STEP 4: official-best dense LOEO checkpoint, score {best['score']:.6f}",
        )
    else:
        print("BIOHUB_DENSE_AUTO_UPLOAD=0: STEP 4 weight upload skipped.")

    history_path, curve_path = save_training_curves(RUN_DIR / "step4_dense_train.log", EXPERIMENT_DIR, STEP4_EPOCHS)
    result = {
        "experiment": EXPERIMENT,
        "step": 4,
        "status": "complete_official_loeo",
        "completed_at": datetime.now().astimezone().isoformat(),
        "split": "126x6bba dense train -> original sparse-GT 12x44b6 official validation",
        "initialization": "scratch",
        "recipe": {
            "epochs": STEP4_EPOCHS,
            "team_md_epochs": 30,
            "epoch_override": "explicit user request on 2026-09-10",
            "batch_size": BATCH_SIZE,
            "lr": LEARNING_RATE,
            "lr_schedule": "cosine",
            "min_lr": MIN_LEARNING_RATE,
            "det_neg_weight": DENSE_NEG_WEIGHT,
            "window_size": WINDOW_SIZE,
            "downsample": [1, 4, 4],
            "max_detections": 0,
            "checkpoint_every": CHECKPOINT_EVERY,
            "seed": SEED,
            "amp": True,
            "gradient_checkpointing": False,
        },
        "checkpoint_table": [metrics[label] for label in sorted(metrics)],
        "official_best": best,
        "sparse_step1_best": 0.6803,
        "improvement_over_sparse": best["score"] - 0.6803,
        "decision_gate": 0.70,
        "decision": "STEP_5_full199_dense" if best["score"] >= 0.70 else "fallback_sparse_neg_weight_0.001",
        "weights_dataset": WEIGHTS_DATASET,
        "weights_folder": STEP4_WEIGHTS_FOLDER,
        "weights_uploaded": AUTO_UPLOAD,
        "training_history": str(history_path.relative_to(PROJECT_ROOT)).replace("\\", "/"),
        "learning_curve": str(curve_path.relative_to(PROJECT_ROOT)).replace("\\", "/") if curve_path else None,
        "leaderboard_submission": False,
    }
    step4_result_path = EXPERIMENT_DIR / "results.json"
    atomic_write_text(step4_result_path, json.dumps(result, ensure_ascii=False, indent=2))
    upsert_results_row(
        "| local Friend A exp03 STEP 4 |",
        f"| local Friend A exp03 STEP 4 | scratch dense LOEO, 126x6bba, {STEP4_EPOCHS}ep (user override from TEAM 30), "
        f"BCE w2 ds1,4,4 cosine 1e-4->1e-5 | - | best **{best['score']:.4f}** ({best_label}; edge "
        f"{best['edge_jaccard']:.4f}, recall {best['node_recall']:.4f}, Npred/Nest {best['n_pred_over_n_est']:.3f}) | GPU | "
        f"12/12 original 44b6; decision `{result['decision']}`; weights `{STEP4_WEIGHTS_FOLDER}/`; no LB |",
    )
    results_md = PROJECT_ROOT / "experiments" / "RESULTS.md"
    context_path = PROJECT_ROOT / "CONTEXT.md"
    update_marked_block(
        context_path,
        "<!-- friend-a-step4-dense-result -->",
        f"- {datetime.now().date().isoformat()} Friend A STEP 4 complete: dense scratch {STEP4_EPOCHS}ep "
        f"(explicit user override from TEAM 30), best official LOEO-44b6 {best['score']:.4f} "
        f"({best_label}, edge {best['edge_jaccard']:.4f}, recall {best['node_recall']:.4f}, "
        f"N_pred/N_est {best['n_pred_over_n_est']:.3f}); next `{result['decision']}`; weights `{STEP4_WEIGHTS_FOLDER}/`; no LB.",
        heading_prefix="## 14. Operational state",
    )
    save_state(
        status="step4_reports_written",
        step4_complete=True,
        step4_best_label=best_label,
        step4_best_score=best["score"],
        step4_best_weights=str(best_weights),
        step4_decision=result["decision"],
    )
    if AUTO_GIT:
        commit_paths = [step4_result_path, history_path, results_md, context_path]
        if curve_path:
            commit_paths.append(curve_path)
        commit_and_push(commit_paths, "Complete Friend A STEP 4 dense LOEO")
    print(f"STEP 4 complete: {best_label} score={best['score']:.6f}; next={result['decision']}")
elif RUN_PIPELINE:
    print(f"Skipping completed STEP 4: score={load_state().get('step4_best_score')}")
else:
    print("Dry preflight: STEP 4 was not started.")


In [ ]:
# Conditional continuation: STEP 5 on a winning dense recipe, otherwise the requested sparse-side fallback.


def write_followup_report(
    *, experiment_dir: Path, result: dict, results_key: str, results_line: str,
    context_marker: str, context_line: str, commit_message: str, extra_paths: list[Path],
) -> None:
    experiment_dir.mkdir(parents=True, exist_ok=True)
    result_path = experiment_dir / "results.json"
    atomic_write_text(result_path, json.dumps(result, ensure_ascii=False, indent=2))
    upsert_results_row(results_key, results_line)
    results_md = PROJECT_ROOT / "experiments" / "RESULTS.md"
    context_path = PROJECT_ROOT / "CONTEXT.md"
    update_marked_block(context_path, context_marker, context_line, heading_prefix="## 14. Operational state")
    if AUTO_GIT:
        commit_and_push([result_path, results_md, context_path, *extra_paths], commit_message)


if RUN_PIPELINE and load_state().get("step4_complete") and CONTINUE_AFTER_STEP4:
    step4_score = float(load_state()["step4_best_score"])
    if step4_score >= 0.70 and not load_state().get("step5_complete"):
        STEP5_EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
        remaining_ids = [dataset_id for dataset_id in all_ids if not (TEACHER_DIR / f"{dataset_id}.geff").is_dir()]
        if remaining_ids and not load_state().get("teacher_extension_smoke_complete"):
            smoke_geffs = predict_teacher_chunk(remaining_ids[:2], "teacher_extension_smoke2")
            if len(smoke_geffs) != min(2, len(remaining_ids)):
                raise RuntimeError("Full-199 teacher extension smoke has incomplete coverage")
            guarded_remove_tree(prediction_output("friend_a_dense_teacher", "teacher_extension_smoke2"), PREDICTIONS_ROOT)
            save_state(teacher_extension_smoke_complete=True)
            print("Full-199 teacher extension smoke passed.")
        ensure_teacher(all_ids, smoke_first=False)

        full_stats = STEP5_EXPERIMENT_DIR / "dense_stats_full199.csv"
        stream_command(
            [
                sys.executable, str(BUILDER_PATH), "--data-dir", str(DATA_DIR),
                "--teacher-dir", str(TEACHER_DIR), "--output-dir", str(DENSE_DATA_DIR),
                "--manifest", str(ALL_MANIFEST_PATH), "--manifest-keys", "all",
                "--dedup-radius-um", str(TEACHER_DEDUP_RADIUS_UM), "--stats-out", str(full_stats),
            ],
            cwd=PROJECT_ROOT,
            env=common_env(DATA_DIR),
            log_path=RUN_DIR / "dense_builder_full199.log",
            label="build_dense_full199",
        )
        with full_stats.open(newline="", encoding="utf-8") as handle:
            full_rows = list(csv.DictReader(handle))
        if len(full_rows) != 199:
            raise RuntimeError(f"Full dense stats contain {len(full_rows)}/199 videos")
        if AUTO_UPLOAD and not load_state().get("full199_geff_uploaded"):
            upload_geff_folders(
                {TEACHER_DATASET_FOLDER: TEACHER_DIR, DENSE_DATASET_FOLDER: DENSE_DATA_DIR},
                all_ids,
                "Friend A STEP 5 preparation: extend teacher and dense GEFF labels to all 199 videos",
            )
            save_state(full199_geff_uploaded=True)

        smoke_training("step5_full199_dense", DENSE_DATA_DIR, all_ids, DENSE_NEG_WEIGHT, 0)
        step5_output = run_training(
            "step5_full199_dense", STEP5_METHOD, DENSE_DATA_DIR, all_ids, holdout_ids,
            STEP5_EPOCHS, DENSE_NEG_WEIGHT, 0,
        )
        final_weights = step5_output / f"edge_predictor_epoch_{STEP5_EPOCHS:03d}.pth"
        if not final_weights.is_file():
            raise FileNotFoundError(final_weights)
        if AUTO_UPLOAD:
            upload_weight_pair(
                final_weights,
                STEP5_WEIGHTS_FOLDER,
                f"Friend A STEP 5: dense scratch model on all 199 videos, epoch {STEP5_EPOCHS}",
            )
        history_path, curve_path = save_training_curves(
            RUN_DIR / "step5_full199_dense_train.log", STEP5_EXPERIMENT_DIR, STEP5_EPOCHS
        )
        handoff = (
            f"Friend A STEP 5 готов: dense-модель с нуля обучена на всех 199 видео, {STEP5_EPOCHS} эпох, "
            f"финальный checkpoint загружен в `{WEIGHTS_DATASET}/{STEP5_WEIGHTS_FOLDER}/`. "
            "Честной LOEO-оценки у full199 модели нет. По плану TEAM.md можно сделать один LB probe как secondary "
            "с первым blend weight 0.5; сам ноутбук ничего на leaderboard не отправлял."
        )
        handoff_path = STEP5_EXPERIMENT_DIR / "handoff_to_vitalii.txt"
        atomic_write_text(handoff_path, handoff + "\n")
        result = {
            "experiment": STEP5_EXPERIMENT,
            "step": 5,
            "status": "complete",
            "completed_at": datetime.now().astimezone().isoformat(),
            "trigger_step4_score": step4_score,
            "train_videos": 199,
            "initialization": "scratch",
            "epochs": STEP5_EPOCHS,
            "recipe": "same dense recipe as STEP 4",
            "selected_checkpoint": f"epoch_{STEP5_EPOCHS:03d}",
            "selection_note": "final prescribed epoch; no leaky internal-best claim",
            "weights_sha256": sha256_file(final_weights),
            "weights_dataset": WEIGHTS_DATASET,
            "weights_folder": STEP5_WEIGHTS_FOLDER,
            "weights_uploaded": AUTO_UPLOAD,
            "official_validation": None,
            "leaderboard_submission": False,
            "handoff_message": handoff,
        }
        extras = [full_stats, history_path, handoff_path]
        if curve_path:
            extras.append(curve_path)
        write_followup_report(
            experiment_dir=STEP5_EXPERIMENT_DIR,
            result=result,
            results_key="| local Friend A exp04 STEP 5 |",
            results_line=(
                f"| local Friend A exp04 STEP 5 | dense scratch all199, {STEP5_EPOCHS}ep, same winning STEP 4 recipe | - | - | GPU | "
                f"final epoch {STEP5_EPOCHS}; dataset `{STEP5_WEIGHTS_FOLDER}/`; no honest LOEO after all199; no LB |"
            ),
            context_marker="<!-- friend-a-step5-full199-dense-result -->",
            context_line=(
                f"- {datetime.now().date().isoformat()} Friend A STEP 5 complete: dense scratch on all 199 videos, "
                f"{STEP5_EPOCHS} epochs; final weights `{STEP5_WEIGHTS_FOLDER}/`; no honest LOEO claim and no LB submission."
            ),
            commit_message="Complete Friend A STEP 5 full199 dense training",
            extra_paths=extras,
        )
        save_state(status="complete", step5_complete=True, handoff_message=handoff)
        print("\nMESSAGE FOR VITALII:\n" + handoff)
    elif step4_score < 0.70 and not load_state().get("fallback_complete"):
        FALLBACK_EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
        fallback_milestones = (5, 10)
        smoke_training("fallback_sparse_neg0001", DATA_DIR, train_ids, FALLBACK_NEG_WEIGHT, 1024)
        fallback_output = run_training(
            "fallback_sparse_neg0001", FALLBACK_METHOD, DATA_DIR, train_ids, holdout_ids,
            FALLBACK_EPOCHS, FALLBACK_NEG_WEIGHT, 1024,
        )
        raw_metrics = evaluate_milestones("fallback_sparse_neg0001", fallback_output, fallback_milestones)
        metrics = {label: compact_metric(label, payload) for label, payload in raw_metrics.items()}
        best_label = max(metrics, key=lambda label: (metrics[label]["score"], metrics[label]["epoch"] or -1))
        best = metrics[best_label]
        best_weights = Path(best["weights"])
        if AUTO_UPLOAD:
            upload_weight_pair(
                best_weights,
                FALLBACK_WEIGHTS_FOLDER,
                f"Friend A fallback: sparse neg-weight 0.001 official-best, score {best['score']:.6f}",
            )
        history_path, curve_path = save_training_curves(
            RUN_DIR / "fallback_sparse_neg0001_train.log", FALLBACK_EXPERIMENT_DIR, FALLBACK_EPOCHS
        )
        result = {
            "experiment": FALLBACK_EXPERIMENT,
            "step": "STEP 4 fallback",
            "status": "complete_official_loeo",
            "completed_at": datetime.now().astimezone().isoformat(),
            "trigger_step4_score": step4_score,
            "split": "126x6bba sparse train -> 12x44b6 official validation",
            "initialization": "scratch",
            "epochs": FALLBACK_EPOCHS,
            "det_neg_weight": FALLBACK_NEG_WEIGHT,
            "checkpoint_table": [metrics[label] for label in sorted(metrics)],
            "official_best": best,
            "weights_dataset": WEIGHTS_DATASET,
            "weights_folder": FALLBACK_WEIGHTS_FOLDER,
            "folder_note": "TEAM.md did not prescribe a fallback folder name; the notebook uses this explicit unique name",
            "leaderboard_submission": False,
        }
        extras = [history_path]
        if curve_path:
            extras.append(curve_path)
        write_followup_report(
            experiment_dir=FALLBACK_EXPERIMENT_DIR,
            result=result,
            results_key="| local Friend A exp04 fallback |",
            results_line=(
                f"| local Friend A exp04 fallback | sparse scratch neg-weight 0.001, 126x6bba, 10ep | - | "
                f"best **{best['score']:.4f}** ({best_label}; edge {best['edge_jaccard']:.4f}, recall "
                f"{best['node_recall']:.4f}, Npred/Nest {best['n_pred_over_n_est']:.3f}) | GPU | "
                f"trigger dense STEP4 {step4_score:.4f}<0.70; weights `{FALLBACK_WEIGHTS_FOLDER}/`; no LB |"
            ),
            context_marker="<!-- friend-a-step4-fallback-result -->",
            context_line=(
                f"- {datetime.now().date().isoformat()} Friend A sparse fallback complete: det-neg-weight 0.001, "
                f"10 epochs, best LOEO-44b6 {best['score']:.4f} ({best_label}, recall {best['node_recall']:.4f}, "
                f"N_pred/N_est {best['n_pred_over_n_est']:.3f}); weights `{FALLBACK_WEIGHTS_FOLDER}/`; no LB."
            ),
            commit_message="Complete Friend A sparse neg-weight fallback",
            extra_paths=extras,
        )
        save_state(status="complete", fallback_complete=True)
        print(f"Fallback complete: {best_label}, score={best['score']:.6f}")
    else:
        print("The conditional follow-up is already complete.")
elif RUN_PIPELINE and not CONTINUE_AFTER_STEP4:
    print("STEP 4 finished. Conditional follow-up is paused by BIOHUB_DENSE_CONTINUE_AFTER_STEP4=0.")
elif RUN_PIPELINE:
    raise RuntimeError("STEP 4 is not complete; conditional continuation cannot start")
else:
    print("Dry preflight: conditional follow-up was not started.")
